## Pre-processing

*This requires the SPC's .csv file of hail reports 1955-2024 be uploaded to the current working directory. The file can be found at https://www.spc.noaa.gov/wcm/#data or in the OneDrive linked in the availability statement in the GitHub readme. Additionally, the RADAR_SITES.txt file found in the OneDrive must also be uploaded to the current working directory.*

This notebook handles pre-processing of hail reports and will output an updated file named `hail_v2.csv`. This file will be filtered to only include reports currently defined as severe by NWS (1"+), within the 2011-2020 time period, and with hail reports only between 20 and 60km from the nearest radar site. The last step is done to ensure quality of radar imagery of storms - too far and the imagery becomes coarse, too close and the storm may be on top of the radar.

In [1]:
# Import necessary packages
import pandas as pd
import numpy as np
import geopy
from geopy import distance
from datetime import datetime, timezone

In [7]:
# Approximate locations of all NWS radar sites east of the Rockies
# I couldn't find a file for this anywhere so I had to manually collect the locations myself
radar_sites = pd.DataFrame(columns=['id', 'lat', 'lon'])
with open("RADAR_SITES.txt", 'r', encoding='utf-8') as file:
    for l in file:
        line = l.strip().split(sep=' ')
        radar_sites.loc[len(radar_sites)] = [line[0].upper(), float(line[1]), float(line[2])*-1]

radar_lats = np.radians(radar_sites['lat'].values)
radar_lons = np.radians(radar_sites['lon'].values)
radar_ids = radar_sites['id'].values
# Find closest radar site given coordinates and the list of radars
def closest_radar(lat, lon):
    lat = np.radians(lat)
    lon = np.radians(lon)

    dlat = radar_lats - lat
    dlon = radar_lons - lon

    a = np.sin(dlat / 2)**2 + np.cos(lat) * np.cos(radar_lats) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    distances = 6371 * c  # Earth radius in km

    idx = np.argmin(distances)
    return radar_ids[idx], distances[idx]

display(radar_sites)

,id,lat,lon
0,KFTX,47.45,-111.38
1,KGGW,48.20,-106.62
2,KBLX,45.85,-108.61
3,KRIW,43.06,-108.48
4,KCYS,41.14,-104.80
...,...,...,...
109,KTYX,43.73,-75.66
110,KBOX,41.96,-71.14
111,KCXX,44.50,-73.14
112,KGYX,43.89,-70.24


In [9]:
# --- Find each report's distance to radar and closest radar
# --- Filter by distance to radar

# Requires the SPC's csv file of hail reports 1955-2024 be uploaded to the working directory.
hail_data = pd.read_csv('1955-2024_hail.csv')
print(f"{len(hail_data)} reports")
hail_data = hail_data[hail_data['yr'] > 2010] #start with 2011
print(f"{len(hail_data)} reports")
hail_data = hail_data[hail_data['yr'] <= 2020] #end with 2020
print(f"{len(hail_data)} reports")
hail_data = hail_data[hail_data['mag'] > 0.99] #only include current severe standard
print(f"{len(hail_data)} reports")
hail_data = hail_data.drop(['elat', 'elon', 'len', 'wid', 'f1', 'f2', 'f3', 'f4', 'om', 'tz', 'st', 'stf', 'stn', 'inj', 'fat', 'loss', 'closs', 'ns', 'sn', 'sg'], axis=1)

# Add cols of closest radar and radar distance
radar = []
radar_dist = []
for i, row in enumerate(hail_data.itertuples(index=False)):
  r, rd = closest_radar(row.slat, row.slon)
  radar.append(r)
  radar_dist.append(rd)

  if i % 10000 == 0:
    print(f"{i}/{len(hail_data)} processed ({(i/len(hail_data))*100:.2f}%)")
hail_data['radar'] = radar
hail_data['radar_dist'] = radar_dist

# Nothing too close to radar or too far away
hail_data = hail_data[(hail_data['radar_dist'] > 20) & (hail_data['radar_dist'] < 60)]
print(f"Final size: {len(hail_data)} reports")

# Add column with pandas datetime
hail_data['datetime'] = pd.to_datetime(
    hail_data['date'].astype(str) + ' ' + hail_data['time'].astype(str),
    format='%Y-%m-%d %H:%M:%S')

display(hail_data[:10])
hail_data.to_csv('hail_v2.csv', index=False)

404910 reports
138056 reports
103912 reports
77838 reports
0/77838 processed (0.00%)
10000/77838 processed (12.85%)
20000/77838 processed (25.69%)
30000/77838 processed (38.54%)
40000/77838 processed (51.39%)
50000/77838 processed (64.24%)
60000/77838 processed (77.08%)
70000/77838 processed (89.93%)
Final size: 18666 reports


,yr,mo,dy,date,time,mag,slat,slon,radar,radar_dist,datetime
266881,2011,10,11,2011-10-11,22:15:00,1.00,34.03,-99.48,KFDR,59.612591,2011-10-11 22:15:00
266891,2011,10,12,2011-10-12,17:44:00,1.00,37.19,-93.63,KSGF,21.515580,2011-10-12 17:44:00
266893,2011,10,12,2011-10-12,18:26:00,1.25,36.99,-93.42,KSGF,25.712838,2011-10-12 18:26:00
266894,2011,10,12,2011-10-12,19:03:00,1.00,36.87,-93.30,KSGF,39.729462,2011-10-12 19:03:00
266922,2011,10,18,2011-10-18,13:43:00,1.75,35.26,-86.56,KHTX,59.181724,2011-10-18 13:43:00
266923,2011,10,18,2011-10-18,13:48:00,1.00,35.26,-86.31,KHTX,44.626380,2011-10-18 13:48:00
266924,2011,10,18,2011-10-18,13:50:00,1.00,35.26,-86.30,KHTX,44.188248,2011-10-18 13:50:00
266925,2011,10,18,2011-10-18,13:54:00,1.00,35.28,-86.37,KHTX,49.372304,2011-10-18 13:54:00
266926,2011,10,18,2011-10-18,13:54:00,1.00,35.28,-86.30,KHTX,46.157768,2011-10-18 13:54:00
266928,2011,10,18,2011-10-18,13:59:00,1.00,35.34,-86.23,KHTX,49.979012,2011-10-18 13:59:00
